In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
df = pd.read_csv(CSV_FILE)

print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data\titanic.csv


In [3]:
    print(f"Dataset loaded successfully! Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    df.head() #replaced print(df.head())   # ugly plain text

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Select numeric columns only
numeric_columns = df.select_dtypes(include='number')
# Calculate statistics (.mean, median, std)
#print("Mean:\n", numeric_columns.mean())
#print("\nMedian:\n", numeric_columns.median())
#print("\nStd Dev:\n", numeric_columns.std())

ihatepython = numeric_columns.agg(['mean', 'median', 'std']).round(2)
print(ihatepython)


        PassengerId  Survived  Pclass    Age  SibSp  Parch   Fare
mean         446.00      0.38    2.31  29.70   0.52   0.38  32.20
median       446.00      0.00    3.00  28.00   0.00   0.00  14.45
std          257.35      0.49    0.84  14.53   1.10   0.81  49.69


Step 4
Objective: Count and analyze missing values in the dataset.

What to do:

Count missing values for each column
Calculate the percentage of missing values
Identify which columns have the most missing data
Code template:

In [5]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    print(f"{col}: {missing_count} missing ({missing_percent:.2f}%)")

df.head()


MISSING VALUES ANALYSIS
PassengerId: 0 missing (0.00%)
Survived: 0 missing (0.00%)
Pclass: 0 missing (0.00%)
Name: 0 missing (0.00%)
Sex: 0 missing (0.00%)
Age: 177 missing (19.87%)
SibSp: 0 missing (0.00%)
Parch: 0 missing (0.00%)
Ticket: 0 missing (0.00%)
Fare: 0 missing (0.00%)
Cabin: 687 missing (77.10%)
Embarked: 2 missing (0.22%)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
#step 5
df_features = df.copy()
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)

def categorize_age(age):
    if age < 13:
        return 'Child'
    elif age < 18:
        return 'Teen'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)

print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)

survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

    Age AgeGroup
0  22.0    Adult
1  38.0    Adult
2  26.0    Adult
3  35.0    Adult
4  35.0    Adult
5   NaN   Senior
6  54.0    Adult
7   2.0    Child
8  27.0    Adult
9  14.0     Teen

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0  1.186076

FEATURE DIFFERENTIATION ANALYSIS

Family Size:
  Survived mean: 1.94
  Not Survived mean: 1.88
  Difference: 0.06


In [7]:
# Step 6: Create Classes for JSON Export

class Passenger:
    """Represents a passenger with all their information."""
    def __init__(self, row):
        self.name = row.get('Name', '')
        self.age = row.get('Age', None)
        self.survived = row.get('Survived', None)
        self.pclass = row.get('Pclass', None)
        self.family_size = row.get('FamilySize', None)
        self.is_alone = row.get('IsAlone', None)

    def to_dict(self):
        return {
            'name': self.name,
            'age': self.age,
            'survived': self.survived,
            'pclass': self.pclass,
            'family_size': self.family_size,
            'is_alone': self.is_alone
        }


class TitanicDataset:
    """Represents the entire Titanic dataset with methods for JSON export."""
    def __init__(self, dataframe):
        self.df = dataframe
        self.passengers = [Passenger(row) for _, row in dataframe.iterrows()]

    def get_statistics(self):
        return {
            'total_passengers': len(self.df),
            'survival_rate': float(self.df['Survived'].mean()),
            'average_age': float(self.df['Age'].mean()),
            'missing_values': self.df.isnull().sum().to_dict()
        }

    def export_to_json(self, filepath):
        data = {
            'statistics': self.get_statistics(),
            'passengers': [p.to_dict() for p in self.passengers]
        }
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"Data exported to {filepath}")


# Use it
dataset = TitanicDataset(df_features)
dataset.export_to_json(JSON_FILE)

Data exported to data\titanic_data.json


In [ ]:
# Step 7

print(json.dumps(json_data, indent=2))

# See just the statistics
print(json_data['statistics'])

# See the first 3 passengers
print(json.dumps(json_data['passengers'][:3], indent=2))
df


NameError: name 'json_data' is not defined